<a href="https://colab.research.google.com/github/ihor-stanishevskyi/bio-ai-journey/blob/main/Untitled0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
print("hello")

hello


In [2]:
!pip install -q git+https://github.com/BiomedSciAI/biomed-multi-alignment.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.9/81.9 kB 6.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.7/104.7 kB 10.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 482.8/482.8 kB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 72.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.5/155.5 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.7/184.7 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 MB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.8/136.8 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.1/409.1 kB 36

In [3]:
from fuse.data.tokenizers.modular_tokenizer.op import ModularTokenizerOp
from mammal.examples.protein_solubility.task import ProteinSolubilityTask
from mammal.keys import CLS_PRED, SCORES
from mammal.model import Mammal
CKPT = "ibm/biomed.omics.bl.sm.ma-ted-458m.protein_solubility"
model = Mammal.from_pretrained(CKPT)
model.eval()
tokenizer_op = ModularTokenizerOp.from_pretrained(CKPT)
print("model loaded on", model.device)

Path doesn't exist. Will try to download from hf hub. pretrained_model_name_or_path='ibm/biomed.omics.bl.sm.ma-ted-458m.protein_solubility'


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Attempting to load model from dir: pretrained_model_name_or_path='/root/.cache/huggingface/hub/models--ibm--biomed.omics.bl.sm.ma-ted-458m.protein_solubility/snapshots/ec682e1e1e4dd769440732127294c00811e04775'


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

model loaded on cpu


In [4]:
protein_seq = "NLMKRCTRGFRKLGKCTTLEEEKCKTLYPRGQCTCSDSKMNTHSCDCKSC"
sample_dict = {"protein_seq": protein_seq}
sample_dict = ProteinSolubilityTask.data_preprocessing(sample_dict=sample_dict, protein_sequence_key="protein_seq", tokenizer_op=tokenizer_op, device=model.device)
batch_dict = model.generate([sample_dict], output_scores=True, return_dict_in_generate=True, max_new_tokens=5)
ans = ProteinSolubilityTask.process_model_output(tokenizer_op=tokenizer_op, decoder_output=batch_dict[CLS_PRED][0], decoder_output_scores=batch_dict[SCORES][0])
print(ans)

{'pred': 1, 'not_normalized_scores': tensor(0.9977), 'normalized_scores': tensor(0.9977)}


In [5]:
def solubility(seq):
  d = {"protein_seq": seq}
  d = ProteinSolubilityTask.data_preprocessing(sample_dict=d, protein_sequence_key="protein_seq", tokenizer_op=tokenizer_op, device=model.device)
  out = model.generate([d], output_scores=True, return_dict_in_generate=True, max_new_tokens=5)
  return ProteinSolubilityTask.process_model_output(tokenizer_op=tokenizer_op, decoder_output=out[CLS_PRED][0], decoder_output_scores=out[SCORES][0])

In [6]:
insulin_b = "FVNQHLCGSHLVEALYLVCGERGFFYTPKT"
lysozyme = "KVFGRCELAAAMKRHGLDNYRGYSLGNWVCAAKFESNFNTQATNRNTDGSTDYGILQINSRWWCNDGRTPGSRNLCNIPCSALLSSDITASVNCAKKIVSDGNGMNAWVAWRNRCKGTDVQAWIRGCRL"
collagen_like = "GPPGPPGPPGPPGPPGPPGPPGPPGPPGPPGPPGPPGPPGPP"
print("insulin B  ", solubility(insulin_b))
print("lysozyme   ", solubility(lysozyme))
print("collagen   ", solubility(collagen_like))

insulin B   {'pred': 1, 'not_normalized_scores': tensor(0.8985), 'normalized_scores': tensor(0.8985)}
lysozyme    {'pred': 1, 'not_normalized_scores': tensor(0.9673), 'normalized_scores': tensor(0.9673)}
collagen    {'pred': 1, 'not_normalized_scores': tensor(0.9980), 'normalized_scores': tensor(0.9980)}
